## Support Vector Machine (SVM)

O [SVM](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html) é um modelo que busca encontrar o melhor hiperplano que separa as classes no espaço de características.

Ele pode utilizar diferentes funções de kernel para lidar com relações lineares e não lineares.

### Hiperparâmetros utilizados

- **C**: controla a margem de separação
  - Valores altos → menor margem (mais complexo)
  - Valores baixos → maior margem (mais simples)

- **kernel**: define o tipo de separação
  - `linear`: separação linear
  - `rbf`: separação não linear (default)

### Estratégia

Foi utilizado GridSearchCV com validação cruzada para identificar a melhor combinação de parâmetros, utilizando a métrica ROC AUC.

In [1]:
scenarios = [
    "sem_submodalidade",
    "submodalidade_agrupada",
    "submodalidade_engineered"
]

tuning_results = []

for scenario in scenarios:
    print(f"\n==========================================")
    print(f"🔎 Scenario: {scenario}")
    print(f"==========================================")
    
    import pandas as pd
    import sys
    import os

    # add raiz do projeto
    sys.path.append(os.path.abspath(".."))

    from sklearn.model_selection import GridSearchCV
    from sklearn.preprocessing import StandardScaler
    from imblearn.pipeline import Pipeline
    from imblearn.over_sampling import SMOTE
    from sklearn.svm import SVC, LinearSVC
    from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

    from preprocessing.main_preprocessing import load_and_preprocess
    from utils.experiment_logger import save_results


save_results(tuning_results, file_path="../results/model_results.csv")

display(pd.DataFrame(tuning_results))



🔎 Scenario: sem_submodalidade

🔎 Scenario: submodalidade_agrupada

🔎 Scenario: submodalidade_engineered


""


## BASELINE

In [2]:
# scenarios = [
#     "sem_submodalidade",
#     "submodalidade_agrupada",
#     "submodalidade_engineered"
# ]

# smote_options = [False, True]

# results = []

# for scenario in scenarios:
#     for use_smote in smote_options:

#         print(f"\n🔎 Scenario: {scenario} | SMOTE: {use_smote}")

#         X_train, X_test, y_train, y_test = load_and_preprocess(
#             "../predictive_models/scrdata_202505.csv",
#             scenario=scenario,
#             use_smote=use_smote
#         )

#         steps = []
#         steps.append(('scaler', StandardScaler()))
#         if use_smote:
#             steps.append(('smote', SMOTE(random_state=42)))
#         steps.append(('svc', LinearSVC(class_weight="balanced", random_state=42, dual=False)))

#         model = Pipeline(steps)

#         model.fit(X_train, y_train)

#         y_pred = model.predict(X_test)
#         y_proba = model.decision_function(X_test)

#         results.append({
#             "model": "SVM",
#             "scenario": scenario,
#             "smote": use_smote,
#             "roc_auc": roc_auc_score(y_test, y_proba),
#             "f1": f1_score(y_test, y_pred),
#             "accuracy": accuracy_score(y_test, y_pred),
#             "n_features": X_train.shape[1],
#             "phase": "baseline",
#             "timestamp": pd.Timestamp.now()
#         })


# save_results(results, file_path="../results/experiments.csv")

# display(pd.DataFrame(results))


## GRIDSEARCH

In [3]:
scenarios = [
    "sem_submodalidade",
    "submodalidade_agrupada",
    "submodalidade_engineered"
]

tuning_results = []

for scenario in scenarios:
    print(f"\n==========================================")
    print(f"🔎 Scenario: {scenario}")
    print(f"==========================================")
    
    X_train, X_test, y_train, y_test = load_and_preprocess(
        "../predictive_models/scrdata_202505.csv",
        scenario=scenario,
        use_smote=False
    )
    print(f"Colunas utilizadas para o cenario '{scenario}': {X_train.columns.tolist()}")
    print(f"Total de features: {len(X_train.columns)}")


    param_grid_svm = {
        "smote": [SMOTE(random_state=42), "passthrough"],
        "svc__C": [0.1, 1, 10]
    }


    grid_svm = GridSearchCV(
        estimator=Pipeline([
            ('scaler', StandardScaler()),
            ('smote', SMOTE(random_state=42)),
            ('svc', LinearSVC(class_weight="balanced", random_state=42, dual=False))
        ]),
        param_grid=param_grid_svm,
        cv=5,                
        scoring="roc_auc",
        n_jobs=2
    )

    grid_svm.fit(X_train, y_train)

    print("Best parameters:", grid_svm.best_params_)
    print("Best ROC AUC (CV):", grid_svm.best_score_)


    #? BEST MODEL TEST EVALUATION

    best_svm = grid_svm.best_estimator_

    y_pred = best_svm.predict(X_test)
    y_proba = best_svm.decision_function(X_test)


    #? TUNING (CV)



    cv_results = pd.DataFrame(grid_svm.cv_results_)
    cv_results['smote_used'] = cv_results['param_smote'].apply(lambda x: x != 'passthrough')

    for smote_val in [True, False]:
        sub_results = cv_results[cv_results['smote_used'] == smote_val]
        if not sub_results.empty:
            best_row = sub_results.sort_values('mean_test_score', ascending=False).iloc[0]
            params = best_row['params']

            tuning_results.append({
                "model": "SVM",
                "scenario": scenario,
                "smote": smote_val,
                "phase": "tuning_cv",
                "roc_auc": best_row['mean_test_score'],
                "f1": None,
                "accuracy": None,
                "best_params": str(params),
                "timestamp": pd.Timestamp.now()
            })

            # Re-fit the best model for this group to get test metrics
            best_model_group = grid_svm.estimator.set_params(**params)
            best_model_group.fit(X_train, y_train)

            y_pred = best_model_group.predict(X_test)
            y_proba = best_model_group.decision_function(X_test)

            tuning_results.append({
                "model": "SVM",
                "scenario": scenario,
                "smote": smote_val,
                "phase": "test",
                "roc_auc": roc_auc_score(y_test, y_proba),
                "f1": f1_score(y_test, y_pred),
                "accuracy": accuracy_score(y_test, y_pred),
                "best_params": str(params),
                "timestamp": pd.Timestamp.now()
            })

    save_results(tuning_results, file_path="../results/model_results.csv")

    display(pd.DataFrame(tuning_results))

save_results(tuning_results, file_path="../results/model_results.csv")

display(pd.DataFrame(tuning_results))



🔎 Scenario: sem_submodalidade
Colunas utilizadas para o cenario 'sem_submodalidade': ['numero_de_operacoes', 'a_vencer_ate_90_dias', 'a_vencer_de_91_ate_360_dias', 'a_vencer_de_361_ate_1080_dias', 'a_vencer_de_1081_ate_1800_dias', 'a_vencer_de_1801_ate_5400_dias', 'a_vencer_acima_de_5400_dias', 'carteira_a_vencer', 'operacoes_missing', 'uf_AL', 'uf_AM', 'uf_AP', 'uf_BA', 'uf_CE', 'uf_DF', 'uf_ES', 'uf_GO', 'uf_MA', 'uf_MG', 'uf_MS', 'uf_MT', 'uf_PA', 'uf_PB', 'uf_PE', 'uf_PI', 'uf_PR', 'uf_RJ', 'uf_RN', 'uf_RO', 'uf_RR', 'uf_RS', 'uf_SC', 'uf_SE', 'uf_SP', 'uf_TO', 'cnae_ocupacao_Autônomo', 'cnae_ocupacao_Empregado de empresa privada', 'cnae_ocupacao_Empregado de entidades sem fins lucrativos', 'cnae_ocupacao_Empresário', 'cnae_ocupacao_MEI', 'cnae_ocupacao_Outros', 'cnae_ocupacao_Servidor ou empregado público', 'porte_Até 1 salário mínimo', 'porte_Indisponível', 'porte_Mais de 1 a 2 salários mínimos', 'porte_Mais de 10 a 20 salários mínimos', 'porte_Mais de 2 a 3 salários mínimos', '

,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,SVM,sem_submodalidade,True,tuning_cv,0.767352,NaN,NaN,"{'smote': SMOTE(random_state=42), 'svc__C': 10}",2026-06-06 18:51:55.991055
1,SVM,sem_submodalidade,True,test,0.764985,0.748478,0.708888,"{'smote': SMOTE(random_state=42), 'svc__C': 10}",2026-06-06 18:52:05.735131
2,SVM,sem_submodalidade,False,tuning_cv,0.766382,NaN,NaN,"{'smote': 'passthrough', 'svc__C': 10}",2026-06-06 18:52:05.735882
3,SVM,sem_submodalidade,False,test,0.764086,0.748603,0.708578,"{'smote': 'passthrough', 'svc__C': 10}",2026-06-06 18:52:13.917961



🔎 Scenario: submodalidade_agrupada
Colunas utilizadas para o cenario 'submodalidade_agrupada': ['numero_de_operacoes', 'a_vencer_ate_90_dias', 'a_vencer_de_91_ate_360_dias', 'a_vencer_de_361_ate_1080_dias', 'a_vencer_de_1081_ate_1800_dias', 'a_vencer_de_1801_ate_5400_dias', 'a_vencer_acima_de_5400_dias', 'carteira_a_vencer', 'operacoes_missing', 'uf_AL', 'uf_AM', 'uf_AP', 'uf_BA', 'uf_CE', 'uf_DF', 'uf_ES', 'uf_GO', 'uf_MA', 'uf_MG', 'uf_MS', 'uf_MT', 'uf_PA', 'uf_PB', 'uf_PE', 'uf_PI', 'uf_PR', 'uf_RJ', 'uf_RN', 'uf_RO', 'uf_RR', 'uf_RS', 'uf_SC', 'uf_SE', 'uf_SP', 'uf_TO', 'cnae_ocupacao_Autônomo', 'cnae_ocupacao_Empregado de empresa privada', 'cnae_ocupacao_Empregado de entidades sem fins lucrativos', 'cnae_ocupacao_Empresário', 'cnae_ocupacao_MEI', 'cnae_ocupacao_Outros', 'cnae_ocupacao_Servidor ou empregado público', 'porte_Até 1 salário mínimo', 'porte_Indisponível', 'porte_Mais de 1 a 2 salários mínimos', 'porte_Mais de 10 a 20 salários mínimos', 'porte_Mais de 2 a 3 salários m

,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,SVM,sem_submodalidade,True,tuning_cv,0.767352,NaN,NaN,"{'smote': SMOTE(random_state=42), 'svc__C': 10}",2026-06-06 18:51:55.991055
1,SVM,sem_submodalidade,True,test,0.764985,0.748478,0.708888,"{'smote': SMOTE(random_state=42), 'svc__C': 10}",2026-06-06 18:52:05.735131
2,SVM,sem_submodalidade,False,tuning_cv,0.766382,NaN,NaN,"{'smote': 'passthrough', 'svc__C': 10}",2026-06-06 18:52:05.735882
3,SVM,sem_submodalidade,False,test,0.764086,0.748603,0.708578,"{'smote': 'passthrough', 'svc__C': 10}",2026-06-06 18:52:13.917961
4,SVM,submodalidade_agrupada,True,tuning_cv,0.824700,NaN,NaN,"{'smote': SMOTE(random_state=42), 'svc__C': 10}",2026-06-06 18:56:29.584682
5,SVM,submodalidade_agrupada,True,test,0.823856,0.784413,0.746852,"{'smote': SMOTE(random_state=42), 'svc__C': 10}",2026-06-06 18:56:48.211561
6,SVM,submodalidade_agrupada,False,tuning_cv,0.824707,NaN,NaN,"{'smote': 'passthrough', 'svc__C': 10}",2026-06-06 18:56:48.212493
7,SVM,submodalidade_agrupada,False,test,0.824128,0.785246,0.747509,"{'smote': 'passthrough', 'svc__C': 10}",2026-06-06 18:57:03.702796



🔎 Scenario: submodalidade_engineered
Colunas utilizadas para o cenario 'submodalidade_engineered': ['numero_de_operacoes', 'a_vencer_ate_90_dias', 'a_vencer_de_91_ate_360_dias', 'a_vencer_de_361_ate_1080_dias', 'a_vencer_de_1081_ate_1800_dias', 'a_vencer_de_1801_ate_5400_dias', 'a_vencer_acima_de_5400_dias', 'carteira_a_vencer', 'operacoes_missing', 'uf_AL', 'uf_AM', 'uf_AP', 'uf_BA', 'uf_CE', 'uf_DF', 'uf_ES', 'uf_GO', 'uf_MA', 'uf_MG', 'uf_MS', 'uf_MT', 'uf_PA', 'uf_PB', 'uf_PE', 'uf_PI', 'uf_PR', 'uf_RJ', 'uf_RN', 'uf_RO', 'uf_RR', 'uf_RS', 'uf_SC', 'uf_SE', 'uf_SP', 'uf_TO', 'cnae_ocupacao_Autônomo', 'cnae_ocupacao_Empregado de empresa privada', 'cnae_ocupacao_Empregado de entidades sem fins lucrativos', 'cnae_ocupacao_Empresário', 'cnae_ocupacao_MEI', 'cnae_ocupacao_Outros', 'cnae_ocupacao_Servidor ou empregado público', 'porte_Até 1 salário mínimo', 'porte_Indisponível', 'porte_Mais de 1 a 2 salários mínimos', 'porte_Mais de 10 a 20 salários mínimos', 'porte_Mais de 2 a 3 salári

,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,SVM,sem_submodalidade,True,tuning_cv,0.767352,NaN,NaN,"{'smote': SMOTE(random_state=42), 'svc__C': 10}",2026-06-06 18:51:55.991055
1,SVM,sem_submodalidade,True,test,0.764985,0.748478,0.708888,"{'smote': SMOTE(random_state=42), 'svc__C': 10}",2026-06-06 18:52:05.735131
2,SVM,sem_submodalidade,False,tuning_cv,0.766382,NaN,NaN,"{'smote': 'passthrough', 'svc__C': 10}",2026-06-06 18:52:05.735882
3,SVM,sem_submodalidade,False,test,0.764086,0.748603,0.708578,"{'smote': 'passthrough', 'svc__C': 10}",2026-06-06 18:52:13.917961
4,SVM,submodalidade_agrupada,True,tuning_cv,0.824700,NaN,NaN,"{'smote': SMOTE(random_state=42), 'svc__C': 10}",2026-06-06 18:56:29.584682
5,SVM,submodalidade_agrupada,True,test,0.823856,0.784413,0.746852,"{'smote': SMOTE(random_state=42), 'svc__C': 10}",2026-06-06 18:56:48.211561
6,SVM,submodalidade_agrupada,False,tuning_cv,0.824707,NaN,NaN,"{'smote': 'passthrough', 'svc__C': 10}",2026-06-06 18:56:48.212493
7,SVM,submodalidade_agrupada,False,test,0.824128,0.785246,0.747509,"{'smote': 'passthrough', 'svc__C': 10}",2026-06-06 18:57:03.702796
8,SVM,submodalidade_engineered,True,tuning_cv,0.787685,NaN,NaN,"{'smote': SMOTE(random_state=42), 'svc__C': 10}",2026-06-06 19:00:20.211702
9,SVM,submodalidade_engineered,True,test,0.787359,0.742641,0.708633,"{'smote': SMOTE(random_state=42), 'svc__C': 10}",2026-06-06 19:00:36.196685


,model,scenario,smote,phase,roc_auc,f1,accuracy,best_params,timestamp
0,SVM,sem_submodalidade,True,tuning_cv,0.767352,NaN,NaN,"{'smote': SMOTE(random_state=42), 'svc__C': 10}",2026-06-06 18:51:55.991055
1,SVM,sem_submodalidade,True,test,0.764985,0.748478,0.708888,"{'smote': SMOTE(random_state=42), 'svc__C': 10}",2026-06-06 18:52:05.735131
2,SVM,sem_submodalidade,False,tuning_cv,0.766382,NaN,NaN,"{'smote': 'passthrough', 'svc__C': 10}",2026-06-06 18:52:05.735882
3,SVM,sem_submodalidade,False,test,0.764086,0.748603,0.708578,"{'smote': 'passthrough', 'svc__C': 10}",2026-06-06 18:52:13.917961
4,SVM,submodalidade_agrupada,True,tuning_cv,0.824700,NaN,NaN,"{'smote': SMOTE(random_state=42), 'svc__C': 10}",2026-06-06 18:56:29.584682
5,SVM,submodalidade_agrupada,True,test,0.823856,0.784413,0.746852,"{'smote': SMOTE(random_state=42), 'svc__C': 10}",2026-06-06 18:56:48.211561
6,SVM,submodalidade_agrupada,False,tuning_cv,0.824707,NaN,NaN,"{'smote': 'passthrough', 'svc__C': 10}",2026-06-06 18:56:48.212493
7,SVM,submodalidade_agrupada,False,test,0.824128,0.785246,0.747509,"{'smote': 'passthrough', 'svc__C': 10}",2026-06-06 18:57:03.702796
8,SVM,submodalidade_engineered,True,tuning_cv,0.787685,NaN,NaN,"{'smote': SMOTE(random_state=42), 'svc__C': 10}",2026-06-06 19:00:20.211702
9,SVM,submodalidade_engineered,True,test,0.787359,0.742641,0.708633,"{'smote': SMOTE(random_state=42), 'svc__C': 10}",2026-06-06 19:00:36.196685
